In [2]:
!pip install requests
!pip install pandas


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import io, os, zipfile
import requests
import pandas as pd

os.makedirs("data", exist_ok=True)
URL = "https://download.geonames.org/export/dump/cities1000.zip"   # if https fails, try http

r = requests.get(URL, timeout=120)
r.raise_for_status()
z = zipfile.ZipFile(io.BytesIO(r.content))
name = [n for n in z.namelist() if n.endswith(".txt")][0]

cols = ["geonameid", "name", "asciiname", "alternatenames", "lat", "lon",
        "feature_class", "feature_code", "country", "cc2", "admin1", "admin2",
        "admin3", "admin4", "population", "elevation", "dem", "timezone", "modified"]

# keep_default_na=False matters: the country code "NA" (Namibia) would otherwise become missing
df = pd.read_csv(z.open(name), sep="\t", header=None, names=cols,
                 keep_default_na=False, na_values=[""])
print(len(df), "places")

172046 places


C:\Users\USER\AppData\Local\Temp\ipykernel_19620\733847064.py:18: DtypeWarning: Columns (13: admin4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(z.open(name), sep="\t", header=None, names=cols,


In [4]:
pop = df[["geonameid", "name", "lat", "lon", "country", "admin1",
          "population", "elevation", "dem", "timezone"]].copy()

pop = pop.drop_duplicates("geonameid")
pop = pop[pop["population"] > 0]                       # many tiny places have 0 = unknown
pop = pop[pop["lat"].between(-90, 90) & pop["lon"].between(-180, 180)]

pop.to_csv("data/population_cities.csv", index=False)
print(len(pop), "places with a population")

160223 places with a population


In [5]:
pop = pd.read_csv("data/population_cities.csv", keep_default_na=False, na_values=[""])

print(pop.nlargest(10, "population")[["name", "country", "population"]])
print(pop.groupby("country")["population"].sum().sort_values(ascending=False).head(10))
print("world total in table:", pop["population"].sum())
print(pop[pop["country"] == "KR"]["population"].sum())     # South Korea total

                    name country  population
27230           Shanghai      CN    24874500
28077          Chongqing      CN    22000000
28096            Chengdu      CN    20937757
28163            Beijing      CN    18960744
27864          Guangzhou      CN    18676605
27198           Shenzhen      CN    17494398
20745           Kinshasa      CD    16000000
135593          Istanbul      TR    15701602
105755             Lagos      NG    15388000
158418  Ho Chi Minh City      VN    14002598
country
CN    882771123
IN    394708946
US    275703031
BR    224202640
JP    146816219
RU    131836721
MX    114409847
DE     90440910
PK     90243597
ID     85989953
Name: population, dtype: int64
world total in table: 4577181558
47941254


In [6]:
import os, requests

os.makedirs("data/grid", exist_ok=True)
URL = "https://worldpop-public-data.soton.ac.uk/GIS/Population/Global_2000_2020/2020/0_Mosaicked/ppp_2020_1km_Aggregated.tif"
PATH = "data/grid/ppp_2020_1km_Aggregated.tif"

def download(url, path):
    if os.path.exists(path):
        print("already have", path); return
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        done = 0
        with open(path + ".part", "wb") as f:
            for chunk in r.iter_content(1024 * 1024):
                f.write(chunk)
                done += len(chunk)
                if total:
                    print(f"\r{done / total:.0%}", end="")
    os.rename(path + ".part", path)
    print("\nsaved", path)

download(URL, PATH)

KeyboardInterrupt: 

In [7]:
pop = df[["geonameid", "name", "lat", "lon", "country", "admin1",
          "population", "elevation", "dem", "timezone"]].copy()

n0 = len(pop)
pop = pop.drop_duplicates("geonameid")
pop = pop[pop["population"] > 0]                                   # 0 means unknown, so drop it
pop = pop[pop["lat"].between(-90, 90) & pop["lon"].between(-180, 180)]
print(f"kept {len(pop)} of {n0} places")

pop.to_csv("data/population_cities.csv", index=False)

kept 160223 of 172046 places


In [8]:
pop = pd.read_csv("data/population_cities.csv", keep_default_na=False, na_values=[""])

print(pop.nlargest(10, "population")[["name", "country", "population"]])
print(pop.groupby("country")["population"].sum().sort_values(ascending=False).head(10))
print("total in table:", f"{pop['population'].sum():,}")
print("South Korea:", f"{pop[pop['country'] == 'KR']['population'].sum():,}")
print("rows with a country code NA (Namibia):", (pop["country"] == "NA").sum())

                    name country  population
27230           Shanghai      CN    24874500
28077          Chongqing      CN    22000000
28096            Chengdu      CN    20937757
28163            Beijing      CN    18960744
27864          Guangzhou      CN    18676605
27198           Shenzhen      CN    17494398
20745           Kinshasa      CD    16000000
135593          Istanbul      TR    15701602
105755             Lagos      NG    15388000
158418  Ho Chi Minh City      VN    14002598
country
CN    882771123
IN    394708946
US    275703031
BR    224202640
JP    146816219
RU    131836721
MX    114409847
DE     90440910
PK     90243597
ID     85989953
Name: population, dtype: int64
total in table: 4,577,181,558
South Korea: 47,941,254
rows with a country code NA (Namibia): 80


In [9]:
import numpy as np
import pandas as pd

pop = pd.read_csv("data/population_cities.csv", keep_default_na=False, na_values=[""])
LAT = np.radians(pop["lat"].values)
LON = np.radians(pop["lon"].values)
P = pop["population"].values
NAMES = pop["name"].values

RINGS = [(0, 20), (20, 50), (50, 100), (100, 200)]        # km from the epicenter

def distance_km(lat, lon):
    la, lo = np.radians(lat), np.radians(lon)
    a = np.sin((LAT - la) / 2) ** 2 + np.cos(la) * np.cos(LAT) * np.sin((LON - lo) / 2) ** 2
    return 2 * 6371 * np.arcsin(np.sqrt(a))

def population_by_ring(lat, lon, rings=RINGS):
    d = distance_km(lat, lon)
    out = []
    for lo_km, hi_km in rings:
        m = (d >= lo_km) & (d < hi_km)
        largest = NAMES[m][np.argmax(P[m])] if m.any() else None
        out.append({"from_km": lo_km, "to_km": hi_km,
                    "population": int(P[m].sum()), "places": int(m.sum()),
                    "largest_place": largest})
    return out

if __name__ == "__main__":
    tests = [("Seoul", 37.57, 126.98), ("Tokyo", 35.68, 139.69), ("Lagos", 6.52, 3.38),
             ("Lima", -12.05, -77.04), ("Fiji, near the date line", -17.8, 179.9),
             ("Mid-Pacific", 0.0, -150.0)]
    for name, la, lo in tests:
        print(name, [r["population"] for r in population_by_ring(la, lo)])

Seoul [13434471, 9244166, 3233795, 12741624]
Tokyo [24290390, 22094205, 9398266, 8909582]
Lagos [19866327, 886273, 1998886, 10823524]
Lima [10020420, 166549, 137255, 1426332]
Fiji, near the date line [0, 0, 8360, 240606]
Mid-Pacific [0, 0, 0, 0]


In [10]:
import os, requests
import pandas as pd

os.makedirs("data", exist_ok=True)
BASE = "https://api.worldbank.org/v2"

def wb(url):
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    js = r.json()
    if not isinstance(js, list) or len(js) < 2 or js[1] is None:
        raise RuntimeError("unexpected response: " + str(js)[:300])
    return js

# 1) Country list, so you can drop regional totals like "World" or "East Asia"
meta = wb(f"{BASE}/country?format=json&per_page=400")
countries = pd.DataFrame([{
    "iso3": c["id"], "iso2": c["iso2Code"], "name": c["name"],
    "region": c["region"]["value"], "income": c["incomeLevel"]["value"],
} for c in meta[1]])
countries = countries[countries["region"] != "Aggregates"]
print(len(countries), "countries and territories")

# 2) GDP per capita, recent years
data = wb(f"{BASE}/country/all/indicator/NY.GDP.PCAP.CD?format=json&per_page=20000&date=2015:2026")
print("pages:", data[0].get("pages"))            # should be 1; if more, add paging
rows = [{"iso3": d["countryiso3code"], "year": int(d["date"]), "gdp_pc": d["value"]}
        for d in data[1] if d["value"] is not None]

# 3) Keep the latest year that has a value, for each country
gdp = pd.DataFrame(rows).sort_values("year").groupby("iso3").tail(1)
vpp = countries.merge(gdp, on="iso3", how="left")
print("countries with no GDP value:", vpp["gdp_pc"].isna().sum())

217 countries and territories
pages: 1
countries with no GDP value: 4


In [11]:
vpp["estimated"] = vpp["gdp_pc"].isna()
vpp["gdp_pc"] = vpp.groupby("income")["gdp_pc"].transform(lambda s: s.fillna(s.median()))
vpp["gdp_pc"] = vpp["gdp_pc"].fillna(vpp["gdp_pc"].median())        # any group with no data at all
print("estimated values:", vpp["estimated"].sum())

estimated values: 4


In [12]:
FACTORS = {"low": 2.0, "base": 3.0, "high": 4.0}    # PLACEHOLDER: replace with Table 3 values or another cited source

In [13]:
for k, f in FACTORS.items():
    vpp[f"value_{k}"] = vpp["gdp_pc"] * f

vpp.to_csv("data/value_per_person.csv", index=False)
print(vpp[["name", "iso2", "gdp_pc", "year", "value_low", "value_base", "value_high"]].head(10))

                   name iso2        gdp_pc    year      value_low  \
0                 Aruba   AW  38590.565029  2024.0   77181.130058   
1           Afghanistan   AF    416.871146  2024.0     833.742293   
2                Angola   AO   3129.476623  2025.0    6258.953247   
3               Albania   AL  12998.147870  2025.0   25996.295739   
4               Andorra   AD  54291.502603  2025.0  108583.005206   
5  United Arab Emirates   AE  50273.512624  2024.0  100547.025249   
6             Argentina   AR  14898.088594  2025.0   29796.177189   
7               Armenia   AM   9474.018493  2025.0   18948.036986   
8        American Samoa   AS  18017.458938  2022.0   36034.917877   
9   Antigua and Barbuda   AG  24819.245468  2025.0   49638.490936   

      value_base     value_high  
0  115771.695087  154362.260115  
1    1250.613439    1667.484585  
2    9388.429870   12517.906493  
3   38994.443609   51992.591478  
4  162874.507809  217166.010412  
5  150820.537873  201094.050497  
6 

In [14]:
for k, f in FACTORS.items():
    vpp[f"value_{k}"] = vpp["gdp_pc"] * f

vpp.to_csv("data/value_per_person.csv", index=False)
print(vpp[["name", "iso2", "gdp_pc", "year", "value_low", "value_base", "value_high"]].head(10))

                   name iso2        gdp_pc    year      value_low  \
0                 Aruba   AW  38590.565029  2024.0   77181.130058   
1           Afghanistan   AF    416.871146  2024.0     833.742293   
2                Angola   AO   3129.476623  2025.0    6258.953247   
3               Albania   AL  12998.147870  2025.0   25996.295739   
4               Andorra   AD  54291.502603  2025.0  108583.005206   
5  United Arab Emirates   AE  50273.512624  2024.0  100547.025249   
6             Argentina   AR  14898.088594  2025.0   29796.177189   
7               Armenia   AM   9474.018493  2025.0   18948.036986   
8        American Samoa   AS  18017.458938  2022.0   36034.917877   
9   Antigua and Barbuda   AG  24819.245468  2025.0   49638.490936   

      value_base     value_high  
0  115771.695087  154362.260115  
1    1250.613439    1667.484585  
2    9388.429870   12517.906493  
3   38994.443609   51992.591478  
4  162874.507809  217166.010412  
5  150820.537873  201094.050497  
6 

In [15]:
vpp = pd.read_csv("data/value_per_person.csv", keep_default_na=False, na_values=[""])
# keep_default_na=False again: Namibia's code "NA" would otherwise become missing

VAL = {}
for k in ("low", "base", "high"):
    m = dict(zip(vpp["iso2"], vpp[f"value_{k}"]))
    s = pop["country"].map(m)
    VAL[k] = s.fillna(s.median()).values          # a place with an unknown country gets the median

def exposed_by_ring(lat, lon, rings=RINGS):
    d = distance_km(lat, lon)
    out = []
    for lo_km, hi_km in rings:
        m = (d >= lo_km) & (d < hi_km)
        out.append({
            "from_km": lo_km, "to_km": hi_km,
            "population": int(P[m].sum()),
            "value_low": float((P[m] * VAL["low"][m]).sum()),
            "value_base": float((P[m] * VAL["base"][m]).sum()),
            "value_high": float((P[m] * VAL["high"][m]).sum()),
        })
    return out

In [16]:
print("country meta:", meta[0])
print("countries after filter:", len(countries))
print("gdp meta:", data[0])
print("gdp rows with a value:", len(rows), "| countries with a value:", gdp["iso3"].nunique())
print("rows in vpp:", len(vpp))
print(pd.read_csv("data/value_per_person.csv", keep_default_na=False).shape)

country meta: {'page': 1, 'pages': 1, 'per_page': '400', 'total': 296}
countries after filter: 217
gdp meta: {'page': 1, 'pages': 1, 'per_page': 20000, 'total': 2926, 'sourceid': '2', 'lastupdated': '2026-11-11'}
gdp rows with a value: 2803 | countries with a value: 258
rows in vpp: 217
(217, 11)


In [17]:
import os, pandas as pd

os.rename("data/vulnerability.csv", "data/vulnerability_appendix1.csv")     # run once

vul = pd.read_csv("data/vulnerability_appendix1.csv", keep_default_na=False)
vul.columns = [c.strip() for c in vul.columns]
print(vul.head(8).to_string())
print("\nISO code lengths:", vul["ISO Code"].astype(str).str.strip().str.len().value_counts().to_dict())
print("all letters:", vul["ISO Code"].astype(str).str.strip().str.isalpha().all())
print("\nStatus values:\n", vul["Status"].value_counts().head(10))
print("\nKorea rows:\n", vul[vul["Country"].str.contains("Korea", case=False)].to_string())

FileExistsError: [WinError 183] Cannot create a file when that file already exists: 'data/vulnerability.csv' -> 'data/vulnerability_appendix1.csv'

In [18]:
vul = vul.rename(columns={
    "Country": "country_name", "ISO Code": "iso_code", "Theta": "theta", "Beta": "beta",
    "Zeta": "zeta", "Per capita GDP, in USD": "gdp_2009",
    "Exposure correction factor, alpha": "alpha", "Status": "status"})
for c in ["theta", "beta", "zeta", "gdp_2009", "alpha"]:
    vul[c] = pd.to_numeric(vul[c], errors="coerce")
vul["iso_code"] = vul["iso_code"].astype(str).str.strip().str.upper()

# Match to the World Bank table, which has both iso2 (needed for the city table) and iso3
vpp = pd.read_csv("data/value_per_person.csv", keep_default_na=False, na_values=[""])
key = "iso3" if vul["iso_code"].str.len().mode()[0] == 3 else "iso2"
codes = vpp[["iso2", "iso3"]].drop_duplicates()
vul = vul.merge(codes, left_on="iso_code", right_on=key, how="left")

print("rows with no matching country:", int(vul["iso2"].isna().sum()))
print(vul[vul["iso2"].isna()][["country_name", "iso_code"]].to_string())

vul = vul.dropna(subset=["iso2"]).drop_duplicates("iso2")
vul[["iso2", "iso3", "country_name", "theta", "beta", "zeta", "alpha", "status"]].to_csv(
    "data/vulnerability_clean.csv", index=False)
print(len(vul), "countries saved")
print("missing theta/beta/zeta/alpha:", vul[["theta", "beta", "zeta", "alpha"]].isna().sum().to_dict())

NameError: name 'vul' is not defined

In [24]:
vul = pd.read_csv("data/vulnerability.csv", keep_default_na=False, na_values=[""])
vpp = pd.read_csv("data/value_per_person.csv", keep_default_na=False, na_values=[""])

ALPHA_DEFAULT = float(vul["alpha"].median())               # fallback = median of the Appendix itself
print("fallback alpha:", ALPHA_DEFAULT)

vpp["alpha"] = vpp["iso2"].map(dict(zip(vul["iso2"], vul["alpha"])))
vpp["alpha_estimated"] = vpp["alpha"].isna()
vpp["alpha"] = vpp["alpha"].fillna(ALPHA_DEFAULT)
vpp["value_pc"] = vpp["gdp_pc"] * vpp["alpha"]
vpp = vpp.drop(columns=[c for c in ["value_low", "value_base", "value_high"] if c in vpp.columns])

vpp.to_csv("data/value_per_person.csv", index=False)
chk = pd.read_csv("data/value_per_person.csv", keep_default_na=False)
print(chk.columns.tolist())
print(chk[["name", "iso2", "gdp_pc", "alpha", "value_pc"]].head(8))
print("countries using the fallback alpha:", int(chk["alpha_estimated"].astype(bool).sum()))

fallback alpha: 16.5245
['iso3', 'iso2', 'name', 'region', 'income', 'year', 'gdp_pc', 'estimated', 'alpha', 'alpha_estimated', 'value_pc']
                   name iso2        gdp_pc   alpha       value_pc
0                 Aruba   AW  38590.565029  14.341  553427.293079
1           Afghanistan   AF    416.871146  15.065    6280.163819
2                Angola   AO   3129.476623  14.790   46284.959259
3               Albania   AL  12998.147870  14.591  189655.975565
4               Andorra   AD  54291.502603  18.268  991797.169554
5  United Arab Emirates   AE  50273.512624  14.702  739121.182603
6             Argentina   AR  14898.088594  18.087  269461.728407
7               Armenia   AM   9474.018493  14.702  139287.019884
countries using the fallback alpha: 5


In [26]:
vul = pd.read_csv("data/vulnerability.csv", keep_default_na=False, na_values=[""])
vpp = pd.read_csv("data/value_per_person.csv", keep_default_na=False, na_values=[""])

DEFAULT = {"theta": float(vul["theta"].median()),
           "beta": float(vul["beta"].median()),
           "zeta": float(vul["zeta"].median())}

In [27]:
!pip install scipy


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [30]:
import numpy as np
import scipy
from scipy.stats import norm
j = vul[vul["iso2"] == "JP"].iloc[0]
print(norm.cdf(np.log(9 / j["theta"]) / j["beta"]))

0.09004930184043847


In [31]:
from scipy.stats import norm

vul = pd.read_csv("data/vulnerability.csv", keep_default_na=False, na_values=[""])
vpp = pd.read_csv("data/value_per_person.csv", keep_default_na=False, na_values=[""])

# Fallback parameters: medians of Table 4 (my assumption; replace them from Appendix I)
DEFAULT = {"theta": 9.6, "beta": 0.10, "zeta": 1.64}

def place_array(df, col, default):
    s = pop["country"].map(dict(zip(df["iso2"], df[col])))
    return s.fillna(default).values.astype(float)

THETA = place_array(vul, "theta", DEFAULT["theta"])
BETA = place_array(vul, "beta", DEFAULT["beta"])
VALUE_PC = place_array(vpp, "value_pc", float(vpp["value_pc"].median()))
ZETA_BY_COUNTRY = dict(zip(vul["iso2"], vul["zeta"]))

def estimate_loss(lat, lon, ring_mmi, rings=RINGS):
    """ring_mmi: one shaking intensity per ring, in the same order as RINGS."""
    d = distance_km(lat, lon)
    nearest_country = pop["country"].values[np.argmin(d)]
    per_ring, total = [], 0.0
    for (lo, hi), mmi in zip(rings, ring_mmi):
        m = (d >= lo) & (d < hi)
        if not m.any() or mmi < 5.0:
            per_ring.append(0.0); continue
        r = norm.cdf(np.log(min(mmi, 9.0) / THETA[m]) / BETA[m])
        loss = float((P[m] * VALUE_PC[m] * r).sum())
        per_ring.append(loss); total += loss
    zeta = float(ZETA_BY_COUNTRY.get(nearest_country, DEFAULT["zeta"]))
    return {"base": total, "low": total * np.exp(-zeta), "high": total * np.exp(zeta),
            "zeta": zeta, "country": nearest_country, "per_ring": per_ring}

if __name__ == "__main__":
    # The intensities below are MADE-UP inputs, only to confirm the function runs
    print(estimate_loss(35.76, 129.19, [9.0, 8.0, 7.0, 6.0]))

{'base': 145960456118.93552, 'low': np.float64(22116760981.44005), 'high': np.float64(963271917091.5659), 'zeta': 1.887, 'country': 'KR', 'per_ring': [27613778028.427444, 81850503581.84543, 35399136845.85322, 1097037662.8094058]}
